# 5) Weighted ALE
- NightRH is the best model based on full data. This code uses the full data to calculate accumulated local effects.
- Weights for adjustment of cow population and number of test-day records are calculated here.

# 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import math, pickle
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import seaborn as sns
import glob
import os, sys, gc
from pathlib import Path
import ast

In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

In [ ]:
from utils import eval_util_module 
import importlib
importlib.reload(eval_util_module)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)
with urlopen('https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json') as response:
    states = json.load(response)

# remove HI, AK, PR
states['features'] = [
    feat for feat in states['features']
    if feat['id'] not in ['02', '72', '15']
]

# 2. Import Full Data:

## 2-1. Full data

In [ ]:
df = pd.read_parquet('3_output/3_final_herd_detrend_df_full.gzip').drop(columns=['year_norm','year_norm_2'])
df.columns

In [ ]:
## checking stats:
df['id'].nunique(), df['GEOID'].nunique(), df['state_abv'].nunique(), df['geoid_herd'].nunique(), df.shape

## 2-2. Formatting

In [ ]:
## some params:
param = {'random':[40,41,42,43],
         'control':['lon','lat','month_cos','month_sin','lac_dim']}

feat_var = ['tmin','tmax_ssrd','rh_am','ag_wind_2m']
target = 'herd_milk_resid'

quantile = np.concatenate([[0.0,0.01,0.03],np.arange(0.05,0.95,0.03),[0.95,0.97,0.99,1.0]])
print(quantile)

warm_state = ['CA','AZ','NM','TX','KS','OK','MO','AR','LA','KY','TN','MS','AL','GA','FL','SC','NC','VA']

control_var = param['control']
sub_ori_cols = control_var + feat_var
sub_cols = ['night_' + item for item in sub_ori_cols]

In [ ]:
## splitting warm vs. cool states
df.loc[df['state_abv'].isin(warm_state),'div'] = 'warm'
df.loc[~df['state_abv'].isin(warm_state),'div'] = 'cool'

# 3. Model based on full data:

## 3-1. Training model based on full data: 

In [ ]:
## best param
cv_output = pd.read_csv('3_output/4_1_tuning_cv_results.csv',index_col=0)
best_params = ast.literal_eval(cv_output.loc[cv_output['features']=='sp1_night_rham_noppt','best_params'].iloc[0])
best_params['seed'] = param['random'][0]                                             
best_params['objective'] = 'reg:squarederror'                              
num_boost_round = best_params['num_boost_round']
del best_params['num_boost_round'], cv_output
best_params

In [ ]:
## Convert to DMatrix
train_df = xgb.DMatrix(df[sub_ori_cols], label=df[target])

In [ ]:
## model:
model = xgb.train(
        params=best_params,
        dtrain=train_df,  # Training data
        num_boost_round=num_boost_round,
        )

In [ ]:
## when reopening the model:
model = xgb.Booster()
model.load_model(f'3_output/5_xbg_model_full.json')
model.set_param({"device": "cpu"})
model.set_param({'n_jobs':-1})

In [ ]:
## predict:
ptrain = model.predict(train_df)
del train_df

## output:
eval_util_module.evaluate(df[target], ptrain)

In [ ]:
df['p_milk'] = ptrain

In [ ]:
## saving model:
model.save_model("3_output/5_xbg_model_full.json")

In [ ]:
fig, ax = plt.subplots(figsize=(3,2))
(df.groupby(['GEOID','cal_yr','month'])[['herd_milk_resid','p_milk']].mean()
 .plot('herd_milk_resid', 'p_milk', 
       ax=ax, marker='o', markersize=2, linestyle='')
)
ax.set_xlim([-1,1])
ax.set_ylim([-1,1])
plt.show()

## 3-2. Checking county-level r2 for sanity:
- if there are any counties having negative r2, exclude them.

In [ ]:
## calculating county-level r2_score:
from sklearn.metrics import r2_score

plot_df = df.groupby(['state_abv','GEOID']).apply(lambda g: r2_score(g[target], g['p_milk'])).reset_index(name='r2')

In [ ]:
plot_df['r2'].describe()

In [ ]:
## SI county-level predictive skill (r2):
plot_df['nonee'] = np.nan
fig = go.Figure(go.Choropleth( locationmode='geojson-id', geojson=counties,
                              locations=plot_df['GEOID'],
                              z=plot_df['r2'], 
                              colorscale="brwnyl",
                              zmin= 0.84, zmax=0.9,
                              marker_line_width=0.2,
                              colorbar_title=dict(text='r2 score', font_family='Arial', side='top',
                                                  font_size=15)
                             ))
fig10 = go.Choropleth(locationmode='USA-states',locations=plot_df[['state_abv','nonee']].drop_duplicates()['state_abv'],
                    z=plot_df[['state_abv','nonee']].drop_duplicates()['nonee'],
                    colorscale = [[0,'rgba(0,0,0,0)'],[1,'rgba(0,0,0,0)']], showscale=False, 
                         marker_line_width=0.6,marker_line_color='black')   
fig.add_trace(fig10)
fig.update_traces(marker_opacity=1)


fig.update_geos(visible=False, scope='usa',
           showsubunits=True,  subunitcolor='black', subunitwidth=1, resolution=110)
fig.update_layout(paper_bgcolor="rgb(255,255,255,255)")
fig.write_image('3_output/fig/SI_county_level_r2_score.png',scale=3)
fig.show()

In [ ]:
## SI map for warm and cool states based on finalized counties:
## warm and cool states:
color_warm = 'palevioletred'
color_cool = 'skyblue'
color_nodata = "#cccccc"
    

plot_df = df[['state_abv','GEOID','div']].drop_duplicates()

fig = go.Figure()
fig.add_trace(go.Choropleth(
    locations=plot_df.loc[plot_df['div'] == 'cool']['GEOID'],
    z=[1] * len(plot_df.loc[plot_df['div'] == 'cool']['GEOID']),  # dummy z values
    geojson=counties,
    colorscale=[[0, color_cool], [1, color_cool]],
    marker_line_color='black',
    showscale=False,
    name='Cool', showlegend=True
))

fig.add_trace(go.Choropleth(
    locations=plot_df.loc[plot_df['div'] == 'warm']['GEOID'],
    z=[1] * len(plot_df.loc[plot_df['div'] == 'warm']['GEOID']),  # dummy z values
    geojson=counties,
    colorscale=[[0, color_warm], [1, color_warm]],
    marker_line_color='black',
    showscale=False,
    name='Warm', showlegend=True
))

## Add a separate layer with text
label_trace = go.Scattergeo(
    locationmode='USA-states',
    locations=plot_df['state_abv'],
    # text=plot_df['state_abv'],
    mode='text',
    textfont=dict(color='black', size=10),
    showlegend=False
)

## Add it to your choropleth
fig.add_trace(label_trace)
fig.update_geos(visible=False, scope='usa', 
               showsubunits=True, subunitcolor='black', subunitwidth=1)

fig.write_image('3_output/fig/SI_warm_cool_states.png',scale=3)                             
fig.show()

# 4. Calculating weight
- To ensure population-representative responses, we account for unequal number of test-day records per cow and differences between our dataset’s county-level cow population and USDA Census data. But our data and USDA data are very similar.

## 4-1. USDA data

In [ ]:
## opening milk production:
usda_cow = pd.read_csv('1_data/usda_nass_census_milk_cow_population_1997_2002_2007_2012_2017_2022_county_level_21May2025.csv', index_col=0)
usda_cow = (usda_cow[['Year','Geo Level','State ANSI','County ANSI','Data Item','Domain','Value']]
            .copy().rename(columns={'Year':'year','Value':'cow','State ANSI':'state_id','County ANSI':'GEOID'})
           )
print(usda_cow['Geo Level'].unique())
usda_cow = usda_cow.loc[~usda_cow['GEOID'].isnull()].copy().reset_index(drop=True)
usda_cow['state_id'] = usda_cow['state_id'].astype(int).astype(str).str.zfill(2)
usda_cow['GEOID'] = usda_cow['state_id'] + usda_cow['GEOID'].astype(int).astype(str).str.zfill(3)
print('remove D :', usda_cow.loc[usda_cow['cow'] != ' (D)']['GEOID'].nunique(),
     usda_cow['GEOID'].nunique())
usda_cow = usda_cow.loc[usda_cow['cow'] != ' (D)'].copy().reset_index(drop=True)
usda_cow['cow'] = usda_cow['cow'].replace(",","", regex=True).astype('float64')
usda_cow['GEOID'].nunique()

In [ ]:
## averaging over census years:
usda_cow = usda_cow.groupby(['state_id','GEOID'])['cow'].mean().reset_index()

## 4-2. Our data:

In [ ]:
## our samples:
county_cow = df[['div','state_abv','GEOID','id','year']].drop_duplicates()
county_cow['state_id'] = county_cow['GEOID'].str[:2]
county_cow = (county_cow.groupby(['state_id','div','state_abv','GEOID','year'])['id'].nunique()
              .reset_index().rename(columns={'id':'drms_cow'})
             )

## 4-3. Weighting:

In [ ]:
## usda avg cow population fraction
usda_cow = pd.merge(usda_cow, 
                    county_cow.groupby(['div','GEOID'])['drms_cow'].mean().reset_index(),
                    on=['GEOID'], how='outer')
print('# of Counties based on DRMS:', 
      usda_cow.loc[usda_cow['drms_cow'].notnull()]['GEOID'].nunique())
usda_cow = usda_cow.loc[~usda_cow['drms_cow'].isnull()].reset_index(drop=True)

## national and region-specific:
usda_cow['usda_n_frac'] = usda_cow['cow'] / usda_cow['cow'].sum()
usda_cow.loc[usda_cow['div'] == 'warm', 'usda_div_frac'] = (
    usda_cow.loc[usda_cow['div'] == 'warm']['cow'] 
    / usda_cow.loc[usda_cow['div'] == 'warm']['cow'].sum()
)
usda_cow.loc[usda_cow['div'] == 'cool', 'usda_div_frac'] = (
    usda_cow.loc[usda_cow['div'] == 'cool']['cow'] 
    / usda_cow.loc[usda_cow['div'] == 'cool']['cow'].sum()
)
print('regional frac (should be 1) :', usda_cow.groupby(['div'])['usda_div_frac'].sum())


In [ ]:
## avg cow population fraction
county_cow['drms_ysum'] = county_cow.groupby(['year'])['drms_cow'].transform('sum')
county_cow['drms_yfrac'] = county_cow['drms_cow'] / county_cow['drms_ysum']
county_cow['drms_div_ysum'] = county_cow.groupby(['div','year'])['drms_cow'].transform('sum')
county_cow['drms_div_yfrac'] = county_cow['drms_cow'] / county_cow['drms_div_ysum']
print(county_cow.groupby(['year','div'])['drms_div_yfrac'].sum().unique())

In [ ]:
## merging two datasets:
county_cow['drms'] = 'Yes'
usda_cow_new = pd.merge(usda_cow[['GEOID','cow','usda_n_frac','usda_div_frac']],
                         county_cow[['state_abv','state_id','div','GEOID','year',
                                    'drms_cow','drms_yfrac','drms_div_yfrac']], 
                        on=['GEOID'], how='outer')

In [ ]:
## calculating cow fraction:
usda_cow_new['cow_n_weight'] = (usda_cow_new['usda_n_frac'] 
                              / usda_cow_new['drms_yfrac'])
usda_cow_new['cow_weight'] = (usda_cow_new['usda_div_frac'] 
                              / usda_cow_new['drms_div_yfrac'])

In [ ]:
## check:
usda_cow_new['check_div'] = usda_cow_new['cow_weight'] * usda_cow_new['drms_div_yfrac']
print('This output should be close to 1 :', 
      usda_cow_new.loc[usda_cow_new['div'] == 'warm'].groupby(['year'])['check_div'].sum().values,
      usda_cow_new.loc[usda_cow_new['div'] == 'cool'].groupby(['year'])['check_div'].sum().values)


In [ ]:
df = pd.merge(df, usda_cow_new[['GEOID','year','cow',
                                'cow_n_weight','cow_weight',
                                'usda_n_frac','usda_div_frac',
                                'drms_cow','drms_yfrac','drms_div_yfrac']],
              on=['GEOID','year'], how='outer')


In [ ]:
## final weight to adjust test-day records:
df['cow_num_test'] = df.groupby(['GEOID','id'])[target].transform('count')
df['final_n_weight'] = df['cow_n_weight'] / df['cow_num_test']
df['final_weight'] = df['cow_weight'] / df['cow_num_test']

In [ ]:
## saving:
df.to_parquet('3_output/5_final_herd_detrend_df_full_cow_weight.gzip',compression='gzip')
del usda_cow, county_cow, usda_cow_new
gc.collect()

# 5. Accumulated Local Effects: 

## 5-1. Calculating ale

In [ ]:
PAD = 1e-6

quantiles = np.concatenate([[0.0,0.01,0.03],np.arange(0.05,0.95,0.03),[0.95,0.97,0.99,1.0]])
print(quantiles)
mid_quant = (quantiles[:-1] + quantiles[1:]) / 2  # Midpoints of quantile bins

# Variables that don't use quantile-based binning
non_quantile_vars = ['month_cos', 'month_sin', 'lac_dim', 'lat', 'lon']

In [ ]:
ale_df = pd.DataFrame()


for region in ['warm','cool']:
    print(f"{region} {'-' * 47}")
    temp = df.loc[df['div'] == region].copy().reset_index(drop=True)
    
    print(temp.shape)
    
    for var in ['month_cos','month_sin']:
        print(var)
        # ---------- Build bin edges for month ----------
        vals = np.unique(temp[var].astype(float))
        vals[np.isclose(vals, 0.0, atol=1e-12)] = 0.0     # snap near-zero to 0
        vals = np.unique(np.round(vals, 6))               # dedupe with rounding
        vals.sort()
        print(vals)
        
        # edges (put at midpoints; pad the ends)
        edges = np.concatenate([[vals[0] - PAD], (vals[:-1] + vals[1:]) / 2, [vals[-1] + PAD]])

        # compute ale:
        out = compute_ale(temp, var, edges, vals, model, 
                          sub_ori_cols, 'final_weight', region)
        ale_df = pd.concat([ale_df, out], ignore_index=True)
    del temp
    gc.collect()

# 2. Continuous features by region (feat_var)
for region in ['warm', 'cool']:
    print(f"{region} {'-' * 47}")
    temp = df.loc[df['div'] == region].copy().reset_index(drop=True)
    
    print(temp.shape)
    for var in feat_var:
        print(var)
        edges = np.unique([np.quantile(temp[var], q=q, weights=temp['final_weight'], 
                                        method='inverted_cdf') for q in quantiles])
        edges[0] -= PAD
        edges[-1] += PAD
        
        bin_mid = [(edges[i] + edges[i+1]) / 2 for i in range(len(edges) - 1)]
    
        out = compute_ale(temp, var, edges, bin_mid, model, 
                          sub_ori_cols, 'final_weight', region)
        ale_df = pd.concat([ale_df, out], ignore_index=True)
    del temp
    gc.collect()

# Continuous features (National)
for var in ['lon','lat', 'lac_dim']:
    edges = np.unique([np.quantile(df[var], q=q, weights=df['final_n_weight'], 
                                    method='inverted_cdf') for q in quantiles])
    edges[0] -= PAD
    edges[-1] += PAD
    
    bin_mid = [(edges[i] + edges[i+1]) / 2 for i in range(len(edges) - 1)]
    
    out = compute_ale(df, var, edges, bin_mid, model, 
                      sub_ori_cols, 'final_n_weight', 'national')
    ale_df = pd.concat([ale_df, out], ignore_index=True)

# saving:
ale_df.to_csv('3_output/5_1_weighted_ale.csv', index=False)

## 5-2. Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12,6), nrows=2, ncols=4, tight_layout=True, sharey=True)

for idx, div in enumerate(['cool','warm']):
    for idy, var in enumerate(feat_var):
        plot_df = ale_df.loc[(ale_df['div'] == div) & (ale_df['feat_abv'] == var)]
        if div == 'warm':
            color = 'tab:red'
        else:
            color = 'tab:blue'
        
        plot_df.sort_values(by='values').plot('values','ale', ax=ax[idx,idy], color=color)
        ax[idx,idy].axvline(plot_df['values'].iloc[2], color='grey', linestyle='--')
        ax[idx,idy].axvline(plot_df['values'].iloc[-3], color='grey', linestyle='--')
        
        ax[idx,idy].set_title(div + ':' +var)
        ax[idx,idy].legend('')
        
        if var == 'tmin':
            ax[idx,idy].set_xlim([-40,30])
        elif var == 'tmax_ssrd':
            ax[idx,idy].set_xlim([0,120])
        elif var == 'rh_am':
            ax[idx,idy].set_xlim([25,100])
        elif var == 'ag_wind_2m':
            ax[idx,idy].set_xlim(0,5)
        ax[idx,idy].grid(False)